# Module 11 - 실습 3: 골든셋 구성 + 자동 평가

## 학습 목표
- Golden Set(기대 입출력 데이터셋)을 구성할 수 있다
- 자동 평가 러너로 LLM 출력 품질을 정량적으로 측정할 수 있다
- 가중치 기반 점수로 여러 평가 기준을 종합할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/11-quality-assurance.md` (섹션 2.6)

## 개념 요약

**Golden Set 구조**:
```json
{
    "case_id": "analysis_001",
    "input": {"query": "버그를 분석해주세요"},
    "expected_output": {"confidence": 0.85, "issues": [...], ...},
    "evaluation_criteria": [
        {"name": "has_summary", "check_type": "field_exists", "field": "summary", "weight": 1.0},
        {"name": "confidence_ok", "check_type": "value_range", "field": "confidence",
         "min_value": 0.5, "max_value": 1.0, "weight": 0.8}
    ]
}
```

In [ ]:
import sys
sys.path.insert(0, '..')

from common.fake_llm import FakeLLM

print("골든셋 평가 실습 준비 완료")

## 실습 1: Golden Set 정의

3개의 테스트 케이스를 가진 Golden Set을 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: 각 케이스는 case_id, input, expected_output, evaluation_criteria를 가짐
# 힌트 2: check_type: "field_exists", "value_range", "list_min_length" 사용
# 힌트 3: weight는 각 기준의 중요도 (0.0~1.0)

GOLDEN_SET = [
    {
        "case_id": "case_001",
        "description": "NullPointerException 분석",
        "input": {
            "query": "getUser() 메서드에서 NullPointerException이 발생합니다."
        },
        "expected_output": {
            "summary": "null 체크 누락",
            "confidence": 0.85,
            "issues": ["null 체크 누락"],
            "recommendation": "null 체크 추가"
        },
        "evaluation_criteria": [
            # TODO: 4개 평가 기준 정의
            # 1. summary 필드 존재 여부 (weight: 1.0)
            # 2. confidence 0.5~1.0 범위 (weight: 0.8)
            # 3. issues 최소 1개 (weight: 0.5)
            # 4. recommendation 필드 존재 여부 (weight: 0.7)
        ]
    },
    # TODO: case_002, case_003 추가
]

In [ ]:
# 검증 셀
assert len(GOLDEN_SET) >= 1, "최소 1개 케이스가 필요합니다"
for case in GOLDEN_SET:
    assert "case_id" in case, f"case_id가 없습니다: {case}"
    assert "input" in case, f"input이 없습니다: {case['case_id']}"
    assert "evaluation_criteria" in case, f"evaluation_criteria가 없습니다: {case['case_id']}"
    assert len(case["evaluation_criteria"]) >= 1, f"평가 기준이 없습니다: {case['case_id']}"
print(f"✅ 실습 1 완료! {len(GOLDEN_SET)}개 케이스의 Golden Set이 준비되었습니다.")

## 실습 2: 평가 체크 함수 구현

개별 평가 기준을 실행하는 `_run_check` 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: check_type == "field_exists": value가 None이 아니고 비어있지 않음
# 힌트 2: check_type == "value_range": min_value <= float(value) <= max_value
# 힌트 3: check_type == "list_min_length": len(value) >= min_length
# 힌트 4: check_type == "exact_match": value == expected_value

def _run_check(criterion: dict, actual_output: dict) -> bool:
    """개별 평가 기준을 실행합니다.
    
    Args:
        criterion: 평가 기준 딕셔너리
        actual_output: LLM 실제 출력
        
    Returns:
        True (통과) 또는 False (실패)
    """
    check_type = criterion["check_type"]
    field = criterion["field"]
    value = actual_output.get(field)
    
    # TODO: 각 check_type별 구현
    pass

In [ ]:
# 검증 셀
test_output = {"summary": "분석 완료", "confidence": 0.85, "issues": ["이슈1"], "recommendation": "조치"}

assert _run_check({"check_type": "field_exists", "field": "summary"}, test_output) == True
assert _run_check({"check_type": "field_exists", "field": "missing_field"}, test_output) == False
assert _run_check({"check_type": "value_range", "field": "confidence", "min_value": 0.5, "max_value": 1.0}, test_output) == True
assert _run_check({"check_type": "value_range", "field": "confidence", "min_value": 0.9, "max_value": 1.0}, test_output) == False
assert _run_check({"check_type": "list_min_length", "field": "issues", "min_length": 1}, test_output) == True
assert _run_check({"check_type": "list_min_length", "field": "issues", "min_length": 5}, test_output) == False
print("✅ 실습 2 완료! _run_check가 모든 check_type을 처리합니다.")

## 실습 3: 단일 케이스 평가 함수 구현

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: 각 criterion에 대해 _run_check() 호출
# 힌트 2: score = weighted_score / total_weight
# 힌트 3: passed = score >= 0.8 (80% 이상이면 통과)

def evaluate_single_case(case: dict, actual_output: dict) -> dict:
    """단일 Golden Case를 평가합니다.
    
    Returns:
        {"case_id": str, "score": float, "passed": bool, "checks": list}
    """
    criteria = case.get("evaluation_criteria", [])
    checks = []
    total_weight = 0.0
    weighted_score = 0.0
    
    # TODO: 각 criterion 평가
    
    score = weighted_score / total_weight if total_weight > 0 else 0.0
    
    return {
        "case_id": case["case_id"],
        "score": round(score, 3),
        "passed": score >= 0.8,
        "checks": checks,
    }

In [ ]:
# 검증 셀
good_output = {"summary": "null 체크 누락", "confidence": 0.85, "issues": ["null 체크"], "recommendation": "조치"}
eval_result = evaluate_single_case(GOLDEN_SET[0], good_output)

assert "case_id" in eval_result, "case_id가 없습니다"
assert "score" in eval_result, "score가 없습니다"
assert "passed" in eval_result, "passed가 없습니다"
assert 0.0 <= eval_result["score"] <= 1.0, f"score가 0~1 범위여야 합니다: {eval_result['score']}"
assert eval_result["passed"] == True, f"좋은 출력은 통과해야 합니다. 점수: {eval_result['score']}"

print(f"평가 결과: 점수={eval_result['score']:.3f}, 통과={eval_result['passed']}")
print("✅ 실습 3 완료! evaluate_single_case가 올바르게 평가합니다.")

## 실습 4: 전체 Golden Set 평가 실행

In [ ]:
# FakeLLM으로 LLM 시뮬레이션
fake_llm = FakeLLM(responses={
    "NullPointerException": '{"summary": "null 체크 누락", "confidence": 0.85, "issues": ["null 체크 누락"], "recommendation": "조건문 추가"}',
    "default": '{"summary": "일반적인 분석 결과", "confidence": 0.75, "issues": ["확인 필요"], "recommendation": "코드 검토"}',
})

import json
import re

def simple_parse(text):
    try:
        return json.loads(text)
    except:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            return json.loads(m.group())
        return {}

def llm_function(input_data: dict) -> dict:
    """FakeLLM을 호출하여 분석 결과를 반환합니다."""
    query = input_data.get("query", "")
    response = fake_llm.invoke(query).content
    return simple_parse(response)

# TODO: 여기에 코드를 작성하세요
# 힌트: GOLDEN_SET의 각 케이스에 대해 llm_function()을 호출하고 evaluate_single_case()로 평가

all_results = []
for case in GOLDEN_SET:
    # TODO: LLM 호출 후 평가
    pass

if all_results:
    avg_score = sum(r["score"] for r in all_results) / len(all_results)
    passed_count = sum(1 for r in all_results if r["passed"])
    print(f"평균 점수: {avg_score:.3f}")
    print(f"통과: {passed_count}/{len(all_results)}")

In [ ]:
# 검증 셀
assert len(all_results) == len(GOLDEN_SET), f"모든 케이스가 평가되어야 합니다"
for r in all_results:
    assert "score" in r and "passed" in r, f"평가 결과 형식이 올바르지 않습니다: {r}"
print(f"✅ 실습 4 완료! {len(GOLDEN_SET)}개 케이스 평가 완료.")

## 정리

이번 실습에서 배운 내용:
1. Golden Set은 입력 + 기대 출력 + 평가 기준의 모음이다
2. 다양한 `check_type`으로 유연한 평가 기준을 정의할 수 있다
3. 가중치(weight)로 기준별 중요도를 조절하여 종합 점수를 계산한다

## 다음 모듈
- **Module 12**: `module_12_advanced_patterns/` - LangGraph 고급 패턴